# Check Citation Faithfulness in Mistral RAG

Mistral RAG can cite its sources — with the References API (`ReferenceChunk`) or,
as here, by asking the model for **structured citations** via structured outputs.
Either way you get, for each claim, a source id and a quoted span. This notebook
adds the missing half: a deterministic check that each quoted span is actually
**faithful** to the source it is attributed to, before the answer reaches a user.

The failure modes worth catching are the ones that read as authoritative and slip
past an LLM judge:

- **fabricated** — a quoted span that appears in no source document;
- **frankenquote** — every word is real, but the exact span was never written
  contiguously;
- **misattributed** — a real span, but attributed to the wrong document.

The pattern is *cheap deterministic detector → expensive judge*:

1. A **0-token verbatim gate** (no model, no API key) checks each quote appears
   verbatim in the cited document. This alone rejects the three failure modes above
   and runs in CI.
2. An **optional Mistral judge** (LLM-as-a-judge, the same technique as
   `mistral/evaluation/RAG_evaluation.ipynb`) decides whether a *verbatim,
   correctly-attributed* quote actually **supports** the claim — a right quote can
   still be the wrong evidence. The burden of proof is on the citation: it defaults
   to unsupported and fails closed.

The gate is inlined below; its standalone, framework-agnostic version lives at
[`verbatim-citation-gate`](https://github.com/Palo-Alto-AI-Research-Lab/verbatim-citation-gate).

In [ ]:
import re


def normalize(text: str) -> str:
    """Case/typography/whitespace-insensitive form for verbatim matching."""
    text = text.lower()
    text = re.sub(r"[‘’]", "'", text)
    text = re.sub(r"[“”]", '"', text)
    text = re.sub(r"[–—]", "-", text)
    text = re.sub(r"[^a-z0-9%.]+", " ", text)
    return " ".join(text.split())


def gate(quote: str, cited_doc_id: str, docs: dict) -> str:
    """Return 'found' | 'misattributed' | 'not_found'. Fails closed on empty quotes."""
    q = normalize(quote)
    if not q:
        return "not_found"
    cited = docs.get(cited_doc_id)
    if cited is not None and q in normalize(cited):
        return "found"
    if any(q in normalize(t) for d, t in docs.items() if d != cited_doc_id):
        return "misattributed"
    return "not_found"

## 1. Run it offline on structured citations

So the notebook runs in CI with no API key, here is a small document library and a
set of `(claim, document_id, quote)` citations in the shape you would get back from
Mistral structured outputs. One is faithful; the others are the three planted
failure modes.

In [ ]:
# document_id -> text (your RAG library / the docs you passed to Mistral)
DOCS = {
    "doc_0": "Mistral Large 2 has a 128k token context window.",
    "doc_1": "Codestral is optimized for code generation across 80+ programming languages.",
}

# What Mistral structured outputs would return: a quote plus the source it cites.
CITATIONS = [
    {"claim": "Mistral Large 2 supports a 128k context.", "document_id": "doc_0",
     "quote": "128k token context window"},                                    # faithful
    {"claim": "Codestral covers 80+ languages.", "document_id": "doc_0",
     "quote": "optimized for code generation across 80+ programming languages"},  # wrong doc
    {"claim": "Codestral has a 128k context for code.", "document_id": "doc_1",
     "quote": "128k token context window for code"},                            # frankenquote
    {"claim": "Mistral Large 2 runs fully offline.", "document_id": "doc_0",
     "quote": "runs entirely on-device with no network"},                       # fabricated
]

for c in CITATIONS:
    status = gate(c["quote"], c["document_id"], DOCS)
    flag = "OK  " if status == "found" else "FLAG"
    print(f"{flag} [{status:>13}]  {c['claim']}")

`found` citations are safe to surface; `misattributed` and `not_found` (fabricated
or frankenquote) should be flagged or dropped — decided deterministically, for zero
tokens.

## 2. Generate structured citations with Mistral

With an API key, ask Mistral to answer **and** return structured citations, then run
the gate over them. This uses `response_format` for structured outputs and needs
`MISTRAL_API_KEY` (not run in CI).

In [ ]:
# pip install mistralai pydantic
import json
import os

if not os.getenv("MISTRAL_API_KEY"):
    print("Set MISTRAL_API_KEY to run the live example.")
else:
    from mistralai import Mistral
    from pydantic import BaseModel

    class Citation(BaseModel):
        claim: str
        document_id: str
        quote: str

    class CitedAnswer(BaseModel):
        answer: str
        citations: list[Citation]

    client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
    library = "\n".join(f"[{doc_id}] {text}" for doc_id, text in DOCS.items())
    resp = client.chat.parse(
        model="mistral-large-latest",
        messages=[
            {"role": "system", "content": "Answer using only the library. For each claim, cite the document_id "
             "and a short quote copied verbatim from that document."},
            {"role": "user", "content": f"Library:\n{library}\n\nQuestion: What context window does Mistral Large 2 "
             "have, and how many languages does Codestral cover?"},
        ],
        response_format=CitedAnswer,
    )
    cited = resp.choices[0].message.parsed
    print(cited.answer, "\n")
    for c in cited.citations:
        status = gate(c.quote, c.document_id, DOCS)
        flag = "OK  " if status == "found" else "FLAG"
        print(f"{flag} [{status:>13}]  {c.quote!r} -> {c.document_id}")

## 3. Optional: an LLM-as-a-judge for the ambiguous case

The gate settles whether a quote *exists*. Whether a real, correctly-attributed
quote actually **supports** its claim is a judgment call — the same LLM-as-a-judge
pattern as `mistral/evaluation/RAG_evaluation.ipynb`, but with the burden of proof on
the citation (default to unsupported, fail closed). Only quotes that pass the gate
(`found`) need reach the judge, so fabrications cost zero judge calls.

For a ready-made burden-of-proof judge (model-agnostic — plug in
`client.chat.complete`), see
[`verbatim-citation-gate`](https://github.com/Palo-Alto-AI-Research-Lab/verbatim-citation-gate).